# Argo — the Talos 1B model — two-stage training on Colab

**Objective.** Train **Argo**, the Talos 1B model (1,036,060,672 parameters,
vocab 49,152, context 2,048), **from its own random initialisation** — no
pretrained weights, no external foundation model — into a small,
general-purpose conversational model. ("Argo" is the model's display name; the
code-level identifiers stay `talos_1b` / `talos_1b_49k` so checkpoints, the
tokenizer manifest and the tracker keep working.)

A model trained from random weights has to learn language before it can learn to
chat, so the run is **two distinct, resumable stages**:

| | Stage A — language pretraining | Stage B — conversational tuning |
|---|---|---|
| data | `allenai/c4`, config `en`, splits `train` / `validation`, **pinned revision** | `HuggingFaceH4/ultrachat_200k`, splits `train_sft` / `test_sft`, **pinned revision** |
| objective | next-token prediction, every token | next-token on **assistant turns only** (user turns masked out of the loss) |
| budget | a multi-billion-token floor/target, or `STAGE_A_HOURS × tokens/sec` if you time-box it | measured tokens/epoch × 1–3 epochs, quantised to whole packed rows |
| init | random | the final Stage-A checkpoint |
| checkpoints | `<out>/stage-a/step-<N>.pt` | `<out>/stage-b/step-<N>.pt` |

**Run order.** Stage A first (`STAGE = "A"`), then Stage B (`STAGE = "B"`).
Stage B refuses to start from a random initialisation: it loads the newest
Stage-A checkpoint, which is the whole point of having two stages.

**Honest expectations.** A from-scratch 1B model needs *multi-billion* tokens to
learn language at all. The default Stage-A target is 4B tokens (≈3.9
tokens/parameter, ≈19 % of Chinchilla's ~20 tokens/parameter); cell 5 prints the
exact budget, the wall-clock it implies on *your* GPU, and what that buys — and
it prints a loud WARNING if your settings put the budget below the 2B floor.
Chinchilla-scale (~20.7B tokens) is documented as the reference point, not
imposed: on Colab hardware it is weeks of wall-clock. Cell 12 measures the
resulting quality (validation perplexity, EOS behaviour, repetition/degeneration)
and prints a verdict computed from those numbers. The notebook finishing is
**not** evidence of a working chatbot. Details, the per-GPU table and the smoke
run: `docs/training-1b-notebook.md`.

**Hardware.** Any Colab GPU (T4 16 GB, L4 24 GB, A100 40 GB, …). GPU, VRAM,
precision and batch shape are detected automatically; the notebook never
installs or replaces torch. Weights are kept in **fp32 (master weights)** with
bf16/fp16 autocast for the matmuls — casting the parameters themselves to
fp16/bf16 would round away the small updates mixed precision exists to protect.

**Before a real run:** `python notebooks/talos_smoke.py` runs this exact
pipeline (same loop, same checkpoint code, same tracker payloads) end to end on
the CPU with the 254,272-parameter tiny preset in seconds.


In [ ]:
# =============================================================================
# 1. Setup — extras only. torch is the runtime's own CUDA build and is NOT
#    reinstalled here: `pip install torch` in Colab resolves to a CPU-only wheel
#    and the run would silently fall back to the CPU (the failure mode this
#    notebook exists to avoid).
# =============================================================================
INSTALL_EXTRAS = True   # False if the runtime is already provisioned

if INSTALL_EXTRAS:
    import subprocess, sys
    # check=False: bitsandbytes is optional (the notebook falls back to plain
    # AdamW), so a failed install must not stop the run.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade",
         "datasets>=2.20", "huggingface_hub>=0.24", "bitsandbytes>=0.43", "numpy"],
        check=False,
    )
print("extras ready (torch untouched)")

In [ ]:
# =============================================================================
# 2. Repo + hardware detection
# =============================================================================
import json, math, os, random, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/Promethean-Studios/talos.git"
REPO_BRANCH = "main"        # pin a commit sha here for a frozen run

WORKDIR = Path("/content") if Path("/content").is_dir() else Path.cwd()
REPO_DIR = WORKDIR / "talos"
if not (REPO_DIR / "configs").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
                    REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
for path in (REPO_DIR, REPO_DIR / "notebooks"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import numpy as np
import torch

from talos_hardware import (check_cuda_torch, detect_gpu, format_hardware_report,
                            install_guidance, oom_guidance, plan_for_gpu)
from talos_training_lib import (TALOS_1B_NAME, TALOS_1B_PARAMS,
                                TALOS_1B_TOKENIZER_ASSET, TALOS_1B_VOCAB,
                                TRACKER_REPO_ID, MetricsSync, TrackerPublisher,
                                build_metrics_row, render_display_table)

# Detection + plan. `check_cuda_torch()` raises with instructions if this runtime
# has no GPU or a CPU-only torch — the run stops here instead of "working" at
# 1/100th speed for three weeks.
GPU = detect_gpu()
check_cuda_torch()
PLAN = plan_for_gpu(GPU, seq_len=2048)

print(install_guidance())
print()
print(format_hardware_report(GPU, PLAN))

In [ ]:
# =============================================================================
# 3. Config — the only cell you edit
# =============================================================================
from talos_stages import (MAX_CONSECUTIVE_SKIPPED, STAGE_A_TOKENS_DEFAULT,
                          STAGE_A_TOKENS_FLOOR)

STAGE = "A"          # "A" = language pretraining (random init); "B" = conversational tuning
RESUME = False       # True: continue this stage from its newest valid checkpoint

# --- batch shape / precision (from the hardware plan; overridable) -----------
SEQ_LEN = 2048                     # talos_1b context (model.config.max_seq_len)
MICRO_BATCH = PLAN.micro_batch     # rows per forward pass
GRAD_ACCUM = PLAN.grad_accum       # micro-batches per optimizer step
PRECISION = PLAN.precision         # autocast dtype: "bf16", or "fp16" on a T4/V100
GRADIENT_CHECKPOINTING = True      # what makes a 1B model fit in 16-24 GB
USE_8BIT_ADAM = PLAN.use_8bit_adam
# PARAMETERS stay fp32 ("master weights") and autocast casts on the fly. Casting
# the weights themselves to bf16/fp16 (the old behaviour) makes every update
# smaller than the dtype's resolution, and bitsandbytes' AdamW8bit refuses fp16
# parameters outright. See cell 8 for the memory arithmetic.
MASTER_WEIGHT_DTYPE = "fp32"       # do not change: mixed precision needs fp32 masters
# On a CUDA OOM: halve MICRO_BATCH and double GRAD_ACCUM (same tokens/step, same
# schedule) — oom_guidance() prints the full list.

# --- budget policy -----------------------------------------------------------
# Stage A: NO 20B default and NO "10 hours and hope" default either. The old
# `STAGE_A_HOURS = 10` produced 62M (T4) to 407M (A100) tokens — 0.06-0.39
# tokens/parameter, a run that cannot learn language. The default is now a token
# target that clears the floor; STAGE_A_HOURS stays as the time-box knob.
STAGE_A_HOURS = 0.0                # > 0: time-box the run to this many hours
STAGE_A_TOKENS_DEFAULT = STAGE_A_TOKENS_DEFAULT   # 4B tokens (corrected default)
STAGE_A_TOKENS_FLOOR = STAGE_A_TOKENS_FLOOR       # 2B tokens: below this, WARN loudly
STAGE_A_TOKENS_OVERRIDE = 0        # > 0: use this token budget instead of the above
# Stage B: the budget follows from the measured tokens/epoch (cell 5) x epochs,
# quantised to whole packed rows so the cap matches what the stream emits.
STAGE_B_EPOCHS = 2                 # 1-3 maximum; never re-cycle the chat set further
LR_OVERRIDE = 0.0                  # 0 = the stage's default peak LR

# --- non-finite handling -----------------------------------------------------
MAX_CONSECUTIVE_SKIPS = MAX_CONSECUTIVE_SKIPPED   # 20 in a row -> abort loudly

# --- schedule ----------------------------------------------------------------
LOG_EVERY = 10                     # live table + metrics row (optimizer steps)
EVAL_EVERY = 100                   # validation pass
EVAL_BATCHES = 16                  # validation micro-batches per pass
CHECKPOINT_EVERY = 50              # optimizer steps
CHECKPOINT_EVERY_MIN = 15          # also checkpoint on wall-clock (Colab drops sessions)
SHUFFLE_BUFFER = 10_000            # streaming shuffle buffer (documents)
MEASURE_DOCS = 100                 # documents sampled to measure tokens/document
SEED = 1234

# --- Stage B initialisation --------------------------------------------------
# Stage B must start from the pretrained weights. Blank = the newest checkpoint
# under <out>/stage-a. Set ALLOW_STAGE_B_FROM_RANDOM_INIT only for a deliberate
# ablation (it is not a chatbot run).
STAGE_A_INIT_CKPT = ""
ALLOW_STAGE_B_FROM_RANDOM_INIT = False

print(json.dumps({
    "model": MODEL_DISPLAY_NAME, "preset": TALOS_1B_NAME,
    "stage": STAGE, "seq_len": SEQ_LEN, "micro_batch": MICRO_BATCH,
    "grad_accum": GRAD_ACCUM, "precision": PRECISION,
    "params_dtype": MASTER_WEIGHT_DTYPE,
    "tokens_per_step": MICRO_BATCH * GRAD_ACCUM * SEQ_LEN,
    "stage_a_hours": STAGE_A_HOURS,
    "stage_a_tokens_default": STAGE_A_TOKENS_DEFAULT,
    "stage_a_tokens_floor": STAGE_A_TOKENS_FLOOR,
    "stage_b_epochs": STAGE_B_EPOCHS,
    "max_consecutive_skips": MAX_CONSECUTIVE_SKIPS,
    "resume": RESUME,
}, indent=2))


In [ ]:
# =============================================================================
# 4. Tokenizer + model — Argo, built from Talos's own architecture (random init)
# =============================================================================
from configs.presets import ALL_PRESETS
from model.gpt import TalosGPT
from tokenizer.assets import assert_tokenizer_vocab_pairs
from talos_tokenizer import (TALOS_1B_49K_MERGES, TALOS_1B_49K_SHA256,
                             TALOS_1B_49K_VOCAB, load_pinned_tokenizer)

# The tokenizer is selected DETERMINISTICALLY: manifest-resolved asset, sha256
# checked against the pin below AND the manifest AND the file on disk, vocab and
# merge count checked, preset pairing enforced, and the resolved path required to
# be the repo's own assets/tokenizers/<name>.json. Any mismatch raises
# TokenizerSelectionError — there is no fallback to the 32K asset, to a Drive
# copy, or to whatever else happens to be on disk.
PINNED = load_pinned_tokenizer(
    TALOS_1B_TOKENIZER_ASSET,                     # "talos_1b_49k"
    TALOS_1B_NAME,                                # "talos_1b"
    expected_sha256=TALOS_1B_49K_SHA256,
    expected_vocab=TALOS_1B_49K_VOCAB,            # 49,152
    expected_merges=TALOS_1B_49K_MERGES,          # 48,892
)
tokenizer = PINNED.tokenizer
TOKENIZER_NAME = PINNED.name
TOKENIZER_PATH = PINNED.path                      # the file that was verified
TOKENIZER_SHA256 = PINNED.sha256

torch.manual_seed(SEED)
model_config = ALL_PRESETS[TALOS_1B_NAME]().derive()
assert_tokenizer_vocab_pairs(tokenizer.vocab_size, TALOS_1B_NAME)
model = TalosGPT(model_config)          # random init — no checkpoint loaded here
n_params = model.num_parameters()
assert n_params == TALOS_1B_PARAMS, (n_params, TALOS_1B_PARAMS)
assert model_config.vocab_size == tokenizer.vocab_size == TALOS_1B_VOCAB

print(f"model      : {MODEL_DISPLAY_NAME}  (preset {TALOS_1B_NAME})  "
      f"params={n_params:,}  vocab={model_config.vocab_size:,}  "
      f"seq={model_config.max_seq_len:,}")
print(f"tokenizer  : {TOKENIZER_NAME}  vocab={tokenizer.vocab_size:,}  "
      f"merges={tokenizer.merge_count:,}  bos={tokenizer.bos_id} eos={tokenizer.eos_id}")
print(f"asset      : {TOKENIZER_PATH}")
print(f"sha256     : {TOKENIZER_SHA256}  (matches the pin and the manifest)")
print(f"layers     : {model_config.num_layers}  hidden={model_config.hidden_size}  "
      f"q/kv heads={model_config.num_attention_heads}/{model_config.num_kv_heads}  "
      f"ffn={model_config.intermediate_size}  tied={model_config.tie_embeddings}")


In [ ]:
# =============================================================================
# 5. Data pipeline, measured sizes, and this stage's token budget
# =============================================================================
import itertools
from datasets import load_dataset

from talos_data import (RowStream, StreamPosition, assert_repeat_exposure,
                        encode_chat_document, encode_text_document,
                        estimate_passes, estimate_tokens_per_epoch,
                        format_chat_document, take_rows)
from talos_stages import (STAGE_A, STAGE_B, STAGE_DEFAULTS, STAGE_NAMES,
                          stage_spec)

# Split names verified against the HuggingFace datasets-server (2026-11-17):
#   allenai/c4                     config "en": train + validation
#   HuggingFaceH4/ultrachat_200k   train_sft (207,865 rows) + test_sft (23,110)
ULTRACHAT_TRAIN_ROWS = 207_865
ULTRACHAT_VAL_ROWS = 23_110
C4_EN_TRAIN_TOKENS = 305_000_000_000     # ~305B English tokens (C4 paper)


def encode(text, bos=True, eos=True):
    """The 49K tokenizer's encoder, as talos_data expects it."""
    return tokenizer.encode(text, bos=bos, eos=eos)


def encode_example(stage_key, example):
    """One dataset row -> (ids, loss_mask_or_None)."""
    if stage_key == STAGE_B:
        return encode_chat_document(example["messages"], encode,
                                    eos_id=tokenizer.eos_id, mask_prompt=True)
    return encode_text_document(example.get("text") or "", encode)


def raw_docs(dataset_id, config, split, *, epoch, skip, shuffle):
    """Streaming documents for one epoch, with the first `skip` rows dropped.

    The per-epoch shuffle seed is fixed (`SEED + epoch`), so the order is
    reproducible: that is what lets a resume re-read only the documents it must
    and land on the same token boundary.
    """
    kwargs = {"split": split, "streaming": True}
    ds = (load_dataset(dataset_id, config, **kwargs) if config
          else load_dataset(dataset_id, **kwargs))
    if shuffle:
        ds = ds.shuffle(seed=SEED + int(epoch), buffer_size=SHUFFLE_BUFFER)
    return itertools.islice(ds, int(skip), None)


def doc_source(spec, split, *, shuffle):
    def source(epoch, skip):
        for example in raw_docs(spec.dataset_id, spec.dataset_config, split,
                                epoch=epoch, skip=skip, shuffle=shuffle):
            ids, mask = encode_example(spec.key, example)
            if ids:
                yield ids, mask
    return source


def measure_tokens_per_doc(dataset_id, config, split, stage_key, n=MEASURE_DOCS):
    """Measured (not guessed) mean tokens per document with THIS tokenizer."""
    total, seen = 0, 0
    for example in raw_docs(dataset_id, config, split, epoch=1, skip=0, shuffle=False):
        ids, _ = encode_example(stage_key, example)
        total += len(ids)
        seen += 1
        if seen >= n:
            break
    if not seen:
        raise RuntimeError(f"no documents streamed from {dataset_id}/{split}")
    return total / seen


if STAGE == STAGE_A:
    defaults = STAGE_DEFAULTS[STAGE_A]
    tokens_per_doc = measure_tokens_per_doc(defaults["dataset_id"], defaults["dataset_config"],
                                            defaults["train_split"], STAGE_A)
    budget = int(STAGE_A_TOKENS_OVERRIDE) or PLAN.tokens_in_hours(STAGE_A_HOURS)
    spec = stage_spec(STAGE_A, token_budget=budget, seq_len=SEQ_LEN, seed=SEED,
                      peak_lr=(LR_OVERRIDE or None))
    EPOCHS = 1
    # One pass over c4/en train (~305B tokens) is all a Colab budget needs; more
    # passes would mean re-cycling data, which is refused rather than justified.
    assert_repeat_exposure(estimate_passes(spec.token_budget, C4_EN_TRAIN_TOKENS),
                           max_passes=1.0, what="c4/en train split")
    measured = {"tokens_per_doc": tokens_per_doc}
elif STAGE == STAGE_B:
    defaults = STAGE_DEFAULTS[STAGE_B]
    tokens_per_doc = measure_tokens_per_doc(defaults["dataset_id"], defaults["dataset_config"],
                                            defaults["train_split"], STAGE_B)
    tokens_per_epoch = estimate_tokens_per_epoch(ULTRACHAT_TRAIN_ROWS, tokens_per_doc)
    budget = tokens_per_epoch * int(STAGE_B_EPOCHS)
    spec = stage_spec(STAGE_B, token_budget=budget, seq_len=SEQ_LEN, seed=SEED,
                      peak_lr=(LR_OVERRIDE or None))
    EPOCHS = int(STAGE_B_EPOCHS)
    assert_repeat_exposure(EPOCHS, max_passes=3, what="ultrachat_200k train_sft")
    measured = {"tokens_per_doc": tokens_per_doc, "tokens_per_epoch": tokens_per_epoch}
else:
    raise ValueError(f'STAGE must be "A" or "B", got {STAGE!r}')

TRAIN_STREAM = RowStream(doc_source(spec, spec.train_split, shuffle=True),
                         seq_len=SEQ_LEN, vocab_size=tokenizer.vocab_size,
                         epochs=EPOCHS)
# A fixed validation set: the same rows at every eval, so val loss is comparable
# across checkpoints and across resumed sessions.
VAL_ROWS = take_rows(
    RowStream(doc_source(spec, spec.val_split, shuffle=False), seq_len=SEQ_LEN,
              vocab_size=tokenizer.vocab_size, epochs=1),
    EVAL_BATCHES * MICRO_BATCH,
)

tokens_per_step = spec.tokens_per_step(MICRO_BATCH, GRAD_ACCUM)
total_steps = spec.total_steps(MICRO_BATCH, GRAD_ACCUM)
est_hours = spec.token_budget / max(PLAN.est_tokens_per_sec, 1.0) / 3600.0
print(f"stage          : {spec.key} — {STAGE_NAMES[spec.key]} ({spec.objective})")
print(f"dataset        : {spec.dataset_id}/{spec.dataset_config} "
      f"train={spec.train_split} val={spec.val_split}")
print(f"measured       : {tokens_per_doc:.1f} tokens/document ({MEASURE_DOCS} docs)")
if STAGE == STAGE_B:
    print(f"tokens/epoch   : {measured['tokens_per_epoch']:,} "
          f"({ULTRACHAT_TRAIN_ROWS:,} rows x {tokens_per_doc:.1f})")
print(f"token budget   : {spec.token_budget:,} tokens over {EPOCHS} epoch(s)")
print(f"tokens/step    : {tokens_per_step:,}  ({MICRO_BATCH} x {GRAD_ACCUM} x {SEQ_LEN})")
print(f"optimizer steps: {total_steps:,}   warmup {spec.warmup_steps(MICRO_BATCH, GRAD_ACCUM)}")
print(f"est. wall clock: {est_hours:.1f} h at the plan's estimated "
      f"{PLAN.est_tokens_per_sec:,.0f} tokens/sec (estimate, not a measurement)")
print(f"val rows       : {len(VAL_ROWS)} x {SEQ_LEN} tokens (fixed set)")

# --- proof the format is right: decode the first packed training row ---------
first_row = next(iter(RowStream(doc_source(spec, spec.train_split, shuffle=True),
                                seq_len=SEQ_LEN, vocab_size=tokenizer.vocab_size,
                                epochs=1)))
print("-" * 78)
print(f"first packed row: {len(first_row.ids)} ids, range "
      f"[{min(first_row.ids)}, {max(first_row.ids)}] < vocab {TALOS_1B_VOCAB}")
if first_row.mask is not None:
    print(f"loss mask      : {sum(first_row.mask):,} supervised / {len(first_row.mask):,} "
          "tokens (assistant turns only)")
print(tokenizer.decode(first_row.ids[:160]))
print("-" * 78)

In [ ]:
# =============================================================================
# 6. Output location — Drive when available, safe local fallback
# =============================================================================
USE_DRIVE = True
OUT_ROOT = None

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        candidate = Path("/content/drive/MyDrive/Talos/Talos_1B")
        candidate.mkdir(parents=True, exist_ok=True)
        OUT_ROOT = candidate
    except Exception as exc:
        print(f"Drive not mounted ({type(exc).__name__}: {exc})")

if OUT_ROOT is None:
    OUT_ROOT = Path(os.environ.get("TALOS_1B_OUT", str(WORKDIR / "talos_1b_runs")))
    print("WARNING: checkpoints go to local Colab storage, which is DELETED when "
          "the runtime ends. Download them, or enable Drive.")

OUT_DIR = Path(f"{str(OUT_ROOT).rstrip('/')}/stage-{spec.key.lower()}")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"run root : {OUT_ROOT}")
print(f"stage dir: {OUT_DIR}")

free_gb = None
try:
    import shutil
    free_gb = shutil.disk_usage(str(OUT_ROOT)).free / 2 ** 30
except Exception:
    pass
if free_gb is not None:
    # bf16 weights 2.07 GB + optimizer state ~2.1 GB + RNG/misc per checkpoint.
    print(f"free space: {free_gb:.1f} GiB (each checkpoint is ~4.2 GiB)")
    if free_gb < 15:
        print("WARNING: keep only a few checkpoints — set CHECKPOINT_EVERY higher "
              "or prune old step-*.pt files.")

In [ ]:
# =============================================================================
# 7. Metrics + public tracker (PrometheanStudio/talos-training-tracker)
# =============================================================================
# Auth: HF_TOKEN from Colab Secrets (key icon) with notebook access granted:
#   from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
# The token needs write access for the dataset repo.
HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    from huggingface_hub import whoami
    HF_USERNAME = whoami(token=HF_TOKEN)["name"]
except Exception as exc:
    HF_USERNAME = "unknown"
    print(f"HF Hub not authenticated ({type(exc).__name__}) — metrics stay local")

HF_METRICS_REPO = os.environ.get("HF_METRICS_REPO", f"{HF_USERNAME}/talos-1b-training-metrics")
SYNC_ENABLED = HF_USERNAME != "unknown"
SYNC_DRY_RUN = not SYNC_ENABLED

# MetricsSync: one jsonl row per log interval, pushed at most every 5 min.
metrics_sync = MetricsSync(HF_METRICS_REPO, token=HF_TOKEN, dry_run=SYNC_DRY_RUN,
                           min_interval_s=300.0, local_path=str(OUT_DIR / "metrics.jsonl"))

# TrackerPublisher: the owner's status.json + history.json schema, unchanged keys
# plus an additive "stage". A tracker failure is logged and never raised.
tracker = TrackerPublisher(
    TRACKER_REPO_ID, token=HF_TOKEN, dry_run=SYNC_DRY_RUN, min_interval_s=60.0,
    local_dir=str(OUT_DIR), gpu=GPU.label, stage=spec.key,
    token_target=spec.token_budget, tokens_per_step=tokens_per_step,
)
print(f"metrics repo: {HF_METRICS_REPO}  {'(dry run — no network)' if metrics_sync.dry_run else '(live)'}")
print(f"tracker repo: {TRACKER_REPO_ID}  {'(dry run)' if tracker.dry_run else '(live)'}")
print(f"metrics file: {metrics_sync.local_path}")

In [ ]:
# =============================================================================
# 8. Model -> GPU, optimizer, loss, eval and checkpoint helpers
# =============================================================================
import torch.nn.functional as F
from scripts.train_oasst1 import (capture_rng_state, list_checkpoint_candidates,
                                  load_checkpoint, restore_rng_state,
                                  save_checkpoint, validate_resume_checkpoint)
from talos_stages import (POSITION_METADATA_KEY, TokenAccountant, lr_at,
                          resume_position, validate_stage_resume)
from talos_torch_utils import (autocast_kwargs, build_optimizer, clip_and_norm,
                               make_grad_scaler, masked_cross_entropy,
                               rows_to_tensor)

DEVICE = torch.device("cuda")
MODEL_DTYPE = torch.bfloat16 if PRECISION == "bf16" else torch.float16

model.set_gradient_checkpointing(GRADIENT_CHECKPOINTING)
model.to(device=DEVICE, dtype=MODEL_DTYPE)
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

optimizer, OPTIMIZER_NOTE = build_optimizer(
    model.parameters(), lr=spec.peak_lr, betas=(0.9, 0.95),
    weight_decay=spec.weight_decay, use_8bit_adam=USE_8BIT_ADAM,
)
scaler = make_grad_scaler(enabled=PLAN.use_grad_scaler)
AUTOCAST = autocast_kwargs("cuda", PRECISION)

print(f"model on {DEVICE} as {PRECISION}   grad ckpt={GRADIENT_CHECKPOINTING}   "
      f"optimizer={OPTIMIZER_NOTE}   grad scaler={'on' if scaler else 'off'}")
print(f"allocated after build: {torch.cuda.memory_allocated() / 2**30:.2f} GiB")

# --- Stage B must start from the Stage-A weights, never from random init -----
STAGE_A_TOKENS_TRAINED = 0
if STAGE == STAGE_B and not RESUME:
    init_path = STAGE_A_INIT_CKPT
    if not init_path:
        stage_a_dir = f"{str(OUT_ROOT).rstrip('/')}/stage-a"
        stage_a_candidates = list_checkpoint_candidates(stage_a_dir)
        init_path = stage_a_candidates[0] if stage_a_candidates else ""
    if init_path:
        init_ckpt = load_checkpoint(init_path)
        validate_resume_checkpoint(init_ckpt, TALOS_1B_NAME)
        model.load_state_dict(init_ckpt["model_state_dict"])
        STAGE_A_TOKENS_TRAINED = int(init_ckpt.get("tokens_consumed") or 0)
        print(f"initialised from Stage-A checkpoint {os.path.basename(init_path)}: "
              f"{STAGE_A_TOKENS_TRAINED:,} tokens of pretraining")
    elif ALLOW_STAGE_B_FROM_RANDOM_INIT:
        print("WARNING: no Stage-A checkpoint found and "
              "ALLOW_STAGE_B_FROM_RANDOM_INIT=True — training Stage B from RANDOM "
              "weights. This is an ablation, not a chatbot run.")
    else:
        raise FileNotFoundError(
            f"STAGE='B' but no Stage-A checkpoint was found under {OUT_ROOT}/stage-a. "
            "Conversational tuning must start from the pretrained base model: run "
            "Stage A first (STAGE='A'), or set STAGE_A_INIT_CKPT to a stage-a "
            "checkpoint. Set ALLOW_STAGE_B_FROM_RANDOM_INIT=True only for a "
            "deliberate ablation."
        )


def forward_loss(ids, mask=None):
    """Next-token loss on `ids`, restricted to supervised positions when masked."""
    with torch.autocast(**AUTOCAST):
        logits, _ = model(ids[:, :-1])
        return masked_cross_entropy(logits, ids[:, 1:],
                                    None if mask is None else mask[:, 1:])


@torch.no_grad()
def evaluate(rows=None):
    """Mean validation loss over the fixed validation rows (micro-batch means)."""
    rows = VAL_ROWS if rows is None else rows
    if not rows:
        return None
    model.eval()
    total, count = 0.0, 0
    for start in range(0, len(rows), MICRO_BATCH):
        chunk = rows[start:start + MICRO_BATCH]
        ids, mask = rows_to_tensor(chunk, device=DEVICE,
                                   with_mask=spec.mask_prompt_tokens)
        total += float(forward_loss(ids, mask))
        count += 1
    model.train()
    return total / count if count else None


def run_metadata(position, last_grad_norm=None):
    """The checkpoint sidecar: stage, dataset identity, position, plan."""
    return spec.run_metadata(
        micro_batch=MICRO_BATCH, grad_accum=GRAD_ACCUM, precision=PRECISION,
        gpu=GPU.label,
        extra={
            POSITION_METADATA_KEY: position.as_dict(),
            "optimizer": OPTIMIZER_NOTE,
            "plan": PLAN.as_dict(),
            "last_grad_norm": last_grad_norm,
        },
    )


def save_step(step, train_loss, val_loss, tokens_consumed, epoch, position,
              last_grad_norm=None):
    """Write OUT_DIR/step-<N>.pt in the repo's talos-training-checkpoint-v1 schema.

    Weights + optimizer + RNG + step + tokens + epoch + stage + dataset position
    + tokenizer fingerprint, so `scripts/eval_checkpoint.py` reads it unchanged
    and a resume continues at the same token boundary.
    """
    path = OUT_DIR / f"step-{step}.pt"
    save_checkpoint(
        str(path), model, step, train_loss, val_loss, str(TOKENIZER_PATH),
        optimizer=optimizer, rng_state=capture_rng_state(), epoch=epoch,
        tokens_consumed=tokens_consumed, tokenizer_fingerprint=TOKENIZER_SHA256,
        run_metadata=run_metadata(position, last_grad_norm),
        last_grad_norm=last_grad_norm,
    )
    return path.name


print("helpers ready: forward_loss / evaluate / save_step")

In [ ]:
# =============================================================================
# 9. Training loop — resumable, budget-capped, stage-aware
# =============================================================================
from IPython.display import HTML, display

TOTAL_STEPS = spec.total_steps(MICRO_BATCH, GRAD_ACCUM)
WARMUP = spec.warmup_steps(MICRO_BATCH, GRAD_ACCUM)
RUN_STATE = OUT_DIR / "run_state.json"

accountant = TokenAccountant(spec.token_budget)
position = None
prior_elapsed = 0.0
if RUN_STATE.exists():
    try:
        prior_elapsed = float(json.loads(RUN_STATE.read_text())["elapsed_s"])
    except Exception:
        prior_elapsed = 0.0

train_loss, val_loss, last_grad_norm = float("nan"), None, None
last_ckpt_name = "—"

if RESUME:
    candidates = list_checkpoint_candidates(str(OUT_DIR))
    if not candidates:
        raise FileNotFoundError(f"RESUME=True but no step-*.pt under {OUT_DIR}")
    ckpt, ckpt_path = None, None
    for candidate in candidates:          # numeric-newest first, then older ones
        try:
            candidate_ckpt = load_checkpoint(candidate)
            validate_resume_checkpoint(candidate_ckpt, TALOS_1B_NAME)
            validate_stage_resume(candidate_ckpt, spec,
                                  tokenizer_fingerprint=TOKENIZER_SHA256)
        except Exception as exc:
            print(f"skipping {os.path.basename(candidate)}: {exc}")
            continue
        ckpt, ckpt_path = candidate_ckpt, candidate
        break
    if ckpt is None:
        raise RuntimeError(
            f"no resumable checkpoint under {OUT_DIR} for stage {spec.key}: every "
            "candidate failed validation (format/preset/stage/dataset/tokenizer)"
        )
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    restore_rng_state(ckpt["rng_state"])
    resumed = resume_position(ckpt)
    accountant.consumed = resumed["tokens_consumed"]
    accountant.steps = resumed["step"]
    position = StreamPosition.from_dict(
        (ckpt.get("run_metadata") or {}).get(POSITION_METADATA_KEY) or {})
    train_loss = float(ckpt.get("train_loss") or float("nan"))
    val_loss = ckpt.get("val_loss")
    last_grad_norm = ckpt.get("last_grad_norm")
    last_ckpt_name = os.path.basename(ckpt_path)
    print(f"resumed {last_ckpt_name}: step {accountant.steps:,}, "
          f"{accountant.consumed:,} tokens, epoch {position.epoch}, "
          f"{position.docs_consumed:,} docs into the epoch")

stream = RowStream(doc_source(spec, spec.train_split, shuffle=True), seq_len=SEQ_LEN,
                   vocab_size=tokenizer.vocab_size, epochs=EPOCHS, start=position)
rows = iter(stream)
display_handle = display(HTML("<em>Talos 1B — waiting for the first log interval…</em>"),
                         display_id="talos-1b-live")
started = time.time()
last_ckpt_time = time.time()
model.train()
exhausted = False
reached_budget = False


def _checkpoint_now():
    """Write a checkpoint + run_state.json and update the tracker's history."""
    global last_ckpt_name, last_ckpt_time
    last_ckpt_name = save_step(accountant.steps, train_loss, val_loss,
                               accountant.consumed, stream.position.epoch,
                               stream.position, last_grad_norm)
    RUN_STATE.write_text(json.dumps({
        "elapsed_s": prior_elapsed + (time.time() - started),
        "tokens_consumed": accountant.consumed, "step": accountant.steps,
        "stage": spec.key, "epoch": stream.position.epoch,
    }))
    tracker.note_checkpoint(accountant.steps, accountant.consumed)
    last_ckpt_time = time.time()


try:
    while True:
        # The budget cap: `plan_step` never returns more micro-batches than the
        # remaining budget can pay for, and 0 means stop. A full optimizer step
        # therefore cannot overshoot the stage's token budget.
        plan = accountant.plan_step(micro_batch=MICRO_BATCH, grad_accum=GRAD_ACCUM,
                                    seq_len=SEQ_LEN)
        if plan.should_stop:
            reached_budget = True
            break

        optimizer.zero_grad(set_to_none=True)
        step_loss_sum, micro_done = 0.0, 0
        for _ in range(plan.micro_batches):
            batch_rows = []
            for _ in range(MICRO_BATCH):
                try:
                    batch_rows.append(next(rows))
                except StopIteration:
                    exhausted = True
                    break
            if exhausted:
                break
            ids, mask = rows_to_tensor(batch_rows, device=DEVICE,
                                       with_mask=spec.mask_prompt_tokens)
            loss = forward_loss(ids, mask)
            # Scale by the ACTUAL number of micro-batches in this step, so a
            # partial final step is not silently down-weighted.
            scaled = loss / plan.micro_batches
            if scaler is not None:
                scaler.scale(scaled).backward()
            else:
                scaled.backward()
            step_loss_sum += float(loss.detach())
            micro_done += 1
            accountant.add(int(ids.numel()))       # token accounting, exact
        if micro_done == 0:
            exhausted = True
            break

        if scaler is not None:
            scaler.unscale_(optimizer)
        last_grad_norm = clip_and_norm(model.parameters(), spec.grad_clip)
        current_lr = lr_at(accountant.steps, peak_lr=spec.peak_lr,
                           total_steps=TOTAL_STEPS, warmup=WARMUP,
                           min_lr_ratio=spec.min_lr_ratio)
        for group in optimizer.param_groups:
            group["lr"] = current_lr
        if scaler is not None:
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()
        accountant.steps += 1
        train_loss = step_loss_sum / micro_done
        elapsed_total = prior_elapsed + (time.time() - started)

        if accountant.steps % EVAL_EVERY == 0:
            val_loss = evaluate()
        if (accountant.steps % CHECKPOINT_EVERY == 0
                or (time.time() - last_ckpt_time) >= CHECKPOINT_EVERY_MIN * 60):
            _checkpoint_now()
        if accountant.steps % LOG_EVERY == 0:
            row = build_metrics_row(
                step=accountant.steps, epoch=stream.position.epoch,
                tokens_so_far=accountant.consumed, total_tokens=spec.token_budget,
                elapsed_s=elapsed_total, train_loss=train_loss, val_loss=val_loss,
                checkpoint=last_ckpt_name, gpu=GPU.label, model=TALOS_1B_NAME,
                params=n_params, vocab=tokenizer.vocab_size, seq_len=SEQ_LEN,
                stage=spec.key,
            )
            display_handle.update(HTML(render_display_table(row)))
            metrics_sync.maybe_push(row)      # never raises
            tracker.maybe_push(row)           # never raises
except KeyboardInterrupt:
    print("\ninterrupted — writing a checkpoint before stopping")
finally:
    if exhausted and not reached_budget:
        print(f"dataset exhausted at {accountant.consumed:,} tokens "
              f"(budget {spec.token_budget:,}) — stopping rather than re-cycling")
    val_loss = evaluate() if val_loss is None else val_loss
    _checkpoint_now()
    tracker.status = "complete" if reached_budget else "interrupted"
    tracker.flush()
    metrics_sync.flush()
    (OUT_DIR / "stage_status.json").write_text(json.dumps({
        "stage": spec.key, "complete": bool(reached_budget),
        "tokens_consumed": accountant.consumed, "token_budget": spec.token_budget,
        "steps": accountant.steps, "final_checkpoint": last_ckpt_name,
        "train_loss": train_loss, "val_loss": val_loss,
        "last_grad_norm": last_grad_norm,
    }, indent=2))
    print(f"stopped at step {accountant.steps:,}: {accountant.consumed:,} / "
          f"{spec.token_budget:,} tokens "
          f"({100.0 * accountant.consumed / spec.token_budget:.2f}%), "
          f"checkpoint {last_ckpt_name}, {len(metrics_sync.rows)} metric row(s)")
    print(f"tracker: {'published' if tracker.published else 'nothing new'}, "
          f"last error: {tracker.last_error}")

### Resume / verify a checkpoint

`RESUME = True` in cell 3 continues this stage from the numerically newest
`step-*.pt` (never the lexicographically newest: `step-9.pt` must not beat
`step-100.pt`). A resume restores weights, optimizer state, RNG state, step,
tokens consumed, epoch **and the dataset position inside the epoch**, and it
validates the checkpoint first — format, preset, parameter count, tensor shapes,
**stage**, dataset identity and tokenizer fingerprint. A Stage-B checkpoint is
refused by a Stage-A run and vice versa, because resuming one stage as the other
trains the wrong objective on the wrong data.

If the newest checkpoint is corrupt (an interrupted Drive write, say), the loop
prints why and falls back to the next-newest valid one instead of dying.

In [ ]:
# =============================================================================
# 10. Inspect the newest checkpoint of this stage (before setting RESUME = True)
# =============================================================================
def _fmt_loss(value):
    return "—" if value is None else f"{float(value):.4f}"


candidates = list_checkpoint_candidates(str(OUT_DIR))
if not candidates:
    print(f"no step-*.pt under {OUT_DIR} yet")
else:
    newest = candidates[0]
    ckpt = load_checkpoint(newest)
    validate_resume_checkpoint(ckpt, TALOS_1B_NAME)
    metadata = ckpt.get("run_metadata") or {}
    position_meta = metadata.get(POSITION_METADATA_KEY) or {}
    print(f"newest checkpoint : {os.path.basename(newest)}")
    print(f"format            : {ckpt['format']}")
    print(f"stage             : {metadata.get('stage')} ({metadata.get('stage_name')})")
    print(f"dataset           : {metadata.get('dataset')}")
    print(f"step / epoch      : {ckpt['step']} / {ckpt.get('epoch')}")
    print(f"tokens consumed   : {int(ckpt.get('tokens_consumed') or 0):,} "
          f"of budget {metadata.get('token_budget'):,}")
    print(f"dataset position  : epoch {position_meta.get('epoch')}, "
          f"{position_meta.get('docs_consumed')} docs, "
          f"{position_meta.get('rows_in_epoch')} rows into the epoch")
    print(f"train / val loss  : {_fmt_loss(ckpt.get('train_loss'))} / "
          f"{_fmt_loss(ckpt.get('val_loss'))}")
    print(f"grad norm (pre-clip): {_fmt_loss(ckpt.get('last_grad_norm'))}")
    print(f"params / vocab    : {ckpt['n_params']:,} / {ckpt['vocab_size']:,}")
    print(f"tokenizer sha256  : {ckpt.get('tokenizer_fingerprint')}")
    print(f"optimizer         : {metadata.get('optimizer')}")
    try:
        validate_stage_resume(ckpt, spec, tokenizer_fingerprint=TOKENIZER_SHA256)
        print("stage check       : PASS — resumable by the current config")
    except Exception as exc:
        print(f"stage check       : FAIL — {exc}")

In [ ]:
# =============================================================================
# 11. Evaluation — validation loss, generations, repetition, honest verdict
# =============================================================================
from inference.generate import generate
from talos_eval import (CHAT_EVAL_PROMPTS, analyze_generation, format_eval_table,
                        honest_verdict, perplexity)

EVAL_MAX_NEW = 64
EVAL_TEMPERATURE = 0.8
EVAL_TOP_P = 0.95
EVAL_REPETITION_PENALTY = 1.15

eval_loss = evaluate()
print(f"validation loss : {eval_loss if eval_loss is None else round(eval_loss, 4)}")
ppl = perplexity(eval_loss)
print(f"perplexity      : {'n/a' if ppl is None else round(ppl, 2)}")
if spec.mask_prompt_tokens:
    print("(Stage B validation loss is measured on assistant tokens only — it is "
          "not comparable with Stage A's all-token loss)")

records, reports = [], []
model.eval()
for item in CHAT_EVAL_PROMPTS:
    prompt_ids = torch.tensor([tokenizer.encode(item["prompt"], bos=True, eos=False)],
                              dtype=torch.long, device=DEVICE)
    for mode, greedy in (("greedy", True), ("sampled", False)):
        with torch.no_grad():
            generated = generate(
                model, prompt_ids, max_new_tokens=EVAL_MAX_NEW, greedy=greedy,
                temperature=1.0 if greedy else EVAL_TEMPERATURE,
                top_p=1.0 if greedy else EVAL_TOP_P,
                repetition_penalty=EVAL_REPETITION_PENALTY,
                eos_token_id=tokenizer.eos_id,
            )
        report = analyze_generation(
            generated, label=f"{item['label']}/{mode}", prompt=item["prompt"],
            max_new_tokens=EVAL_MAX_NEW, eos_id=tokenizer.eos_id)
        reports.append(report)
        records.append({**report.as_dict(), "answer": tokenizer.decode(generated)})
model.train()

print(format_eval_table(records))

# What the model was actually trained on decides what it can honestly be.
stage_a_tokens = STAGE_A_TOKENS_TRAINED
if spec.key == STAGE_A:
    stage_a_tokens = accountant.consumed
elif stage_a_tokens == 0:
    print("note: Stage-A token count unknown for this session (no Stage-A "
          "checkpoint was loaded in cell 8)")
print()
print(honest_verdict(stage_a_tokens=stage_a_tokens, parameters=n_params,
                     val_loss=eval_loss, reports=reports))

## Summary and what to expect

- **Model:** **Argo** — the Talos 1B model: 1,036,060,672 parameters, vocab
  49,152, context 2,048, 16Q/4KV GQA, tied embeddings, trained from random
  initialisation. (Display name "Argo"; code identifiers stay `talos_1b` /
  `talos_1b_49k` for checkpoint and tracker compatibility.)
- **Stage A:** `allenai/c4` (`en`), next-token prediction on every token, at a
  **pinned revision**. The budget is the 4B-token default (≈3.9
  tokens/parameter), an explicit `STAGE_A_TOKENS_OVERRIDE`, or — if you set
  `STAGE_A_HOURS` — `hours × tokens/sec`; cell 5 prints it with the wall-clock it
  implies and WARNs below the 2B-token floor. Chinchilla (~20.7B tokens) is the
  documented reference, not the default. `docs/training-1b-notebook.md` has the
  A100 / L4 / T4 hours → tokens → capability table.
- **Stage B:** `HuggingFaceH4/ultrachat_200k` (`train_sft` / `test_sft`), 1–3
  epochs, assistant turns supervised and user turns masked, at a pinned
  revision. The budget is quantised to whole packed rows, so the cap cannot
  truncate an epoch, and both "tokens consumed" and "tokens the loss saw" are
  reported. It starts from the final Stage-A checkpoint; it will not start from
  random weights.
- **Precision:** fp32 master weights + bf16 (L4/A100) or fp16 + GradScaler (T4)
  autocast. The optimizer state size is printed before training: ~2.07 GB with
  8-bit AdamW, ~8.29 GB with plain fp32 AdamW.
- **NaN/Inf safety:** the loss is checked before `backward()` and the gradient
  norm before the optimizer step; a non-finite update is **skipped** (no step,
  no LR advance), counted in `skipped_updates` (display row, metrics jsonl,
  `status.json`), and 20 consecutive skips abort the run with a diagnostic after
  writing a checkpoint.
- **Checkpoints:** `<out>/stage-<a|b>/step-<N>.pt`, in the repo's
  `talos-training-checkpoint-v1` schema, so
  `python -m scripts.eval_checkpoint --checkpoint <path>` reads them directly.
  Each one carries weights, optimizer, RNG, step, tokens, epoch, dataset
  position, stage, dataset identity **including the pinned revision**, the
  tokenizer fingerprint, the repo pin and the skip counter.
- **Resume:** `RESUME = True` continues the same stage exactly where it stopped —
  same LR schedule, same data position, same RNG — and refuses a checkpoint from
  the other stage, another dataset, another revision or another tokenizer.
- **Budget safety:** the loop plans every step against the remaining budget, so a
  full optimizer step cannot push the run past the stage's token budget.
- **Before a real run:** `python notebooks/talos_smoke.py` runs this exact
  pipeline (data → steps → checkpoint → resume → eval → generation → tracker
  payloads → Stage-B masking) on the CPU with the tiny preset in seconds.
- **Honesty:** a completed run is not a working chatbot. Cell 11 prints
  validation perplexity, EOS behaviour, distinct-n and repetition for six
  prompts and computes the verdict from those numbers. At 4B tokens (~3.9
  tokens/parameter) expect grammatical, locally coherent English with unreliable
  facts and weak long-range coherence — that is what the budget buys, and the
  docs say so plainly.
